# Lab 8 : Additive and Multiplicative attention

### Theory

Neural Machine Translation (NMT) is a deep learning approach that translates text from one language to another using an encoder-decoder architecture. The encoder processes the input sentence and converts it into numerical representations, while the decoder generates the translated sentence one word at a time. Earlier encoder-decoder models used a single fixed-length context vector to represent the entire input sequence, which often led to poor performance when translating long and complex sentences.

The introduction of the Attention Mechanism significantly improved NMT by allowing the decoder to focus on the most relevant parts of the input sentence during translation. Instead of using a single context vector for the entire sentence, attention computes a dynamic context vector for each output word. This enables the model to learn alignments between source and target words automatically and improves translation accuracy, especially for longer sentences.

The encoder commonly employs a bidirectional Recurrent Neural Network (RNN) to capture contextual information from both past and future words. At each decoding step, attention weights are assigned to the encoder's hidden states, indicating the importance of each input word for generating the next output word. The weighted combination of these hidden states forms the context vector used by the decoder.

Further improvements to attention-based NMT introduced Global and Local Attention mechanisms. Global Attention considers all source words while generating each target word, providing comprehensive contextual information. In contrast, Local Attention focuses only on a small subset of source words, reducing computational complexity while maintaining translation quality. These approaches make the model more efficient and effective for handling long sequences.

Attention-based Neural Machine Translation has become a fundamental technique in Natural Language Processing because it improves translation performance, learns meaningful word alignments automatically, and provides the foundation for many modern sequence-to-sequence models.


#### Requirements

In [1]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


#### Loading Data

In [2]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [3]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [4]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [5]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [6]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [7]:
PATH = r'eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words. 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['vous etes fort elegante', 'you re very sophisticated']


15

#### Encoder

In [8]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

#### Decoder

In [9]:
class LuongDotAttention(nn.Module):

    def __init__(self, hidden_size):
        super().__init__()

        self.Wc = nn.Linear(
            hidden_size*2,
            hidden_size
        )


    def forward(self, query, keys):

        # e(t,i)=stT*hi

        scores = torch.bmm(
            query,
            keys.transpose(1,2)
        )


        # softmax

        weights = F.softmax(
            scores,
            dim=-1
        )


        # context vector

        context = torch.bmm(
            weights,
            keys
        )


        # [ct;st]

        combined = torch.cat(
            (context,query),
            dim=-1
        )


        # s~t

        attentional_hidden = torch.tanh(
            self.Wc(combined)
        )


        return attentional_hidden,weights

In [10]:
class LuongAttnDecoderRNN(nn.Module):

    def __init__(self,
                 hidden_size,
                 output_size,
                 dropout_p=0.1):

        super().__init__()

        self.embedding = nn.Embedding(
            output_size,
            hidden_size
        )

        self.dropout = nn.Dropout(
            dropout_p
        )


        # GRU takes only embedding

        self.rnn = nn.RNN(
            hidden_size,
            hidden_size,
            batch_first=True
        )


        self.attention = LuongDotAttention(
            hidden_size
        )


        self.out = nn.Linear(
            hidden_size,
            output_size
        )


    def forward(self,
                encoder_outputs,
                encoder_hidden,
                target_tensor=None):


        batch_size = encoder_outputs.size(0)


        decoder_input = torch.empty(
            batch_size,
            1,
            dtype=torch.long,
            device=device
        ).fill_(SOS_token)


        decoder_hidden = encoder_hidden


        decoder_outputs = []

        attentions = []


        for i in range(MAX_LENGTH):

            decoder_output,\
            decoder_hidden,\
            attn_weights = self.forward_step(

                decoder_input,
                decoder_hidden,
                encoder_outputs

            )


            decoder_outputs.append(
                decoder_output
            )


            attentions.append(
                attn_weights
            )


            # teacher forcing

            if target_tensor is not None:

                decoder_input = target_tensor[:,i].unsqueeze(1)

            else:

                _,topi = decoder_output.topk(1)

                decoder_input = topi.squeeze(-1).detach()


        decoder_outputs = torch.cat(
            decoder_outputs,
            dim=1
        )


        decoder_outputs = F.log_softmax(
            decoder_outputs,
            dim=-1
        )


        attentions = torch.cat(
            attentions,
            dim=1
        )


        return decoder_outputs,\
               decoder_hidden,\
               attentions



    def forward_step(self,
                     input,
                     hidden,
                     encoder_outputs):


        embedded = self.dropout(

            self.embedding(input)

        )


        # GRU first

        output,hidden = self.rnn(

            embedded,
            hidden

        )


        # current hidden state

        query = output


        # multiplicative attention

        attentional_hidden,\
        attn_weights = self.attention(

            query,
            encoder_outputs

        )


        prediction = self.out(
            attentional_hidden
        )


        return prediction,\
               hidden,\
               attn_weights

#### Prepare Training Data

In [11]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

#### Training loop

In [12]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [13]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [14]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [15]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

#### Evaluation Code

In [17]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [18]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

#### Training and Evaluating

In [19]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = LuongAttnDecoderRNN(

            hidden_size,
            output_lang.n_words

          ).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 18s (- 11m 46s) (5 2%) 1.8676
0m 32s (- 10m 8s) (10 5%) 1.1052
0m 46s (- 9m 36s) (15 7%) 0.7358
1m 1s (- 9m 9s) (20 10%) 0.4665
1m 15s (- 8m 46s) (25 12%) 0.3008
1m 29s (- 8m 26s) (30 15%) 0.2102
1m 43s (- 8m 7s) (35 17%) 0.1672
1m 57s (- 7m 49s) (40 20%) 0.1395
2m 11s (- 7m 32s) (45 22%) 0.1185
2m 25s (- 7m 17s) (50 25%) 0.1093
2m 39s (- 7m 1s) (55 27%) 0.1027
2m 54s (- 6m 46s) (60 30%) 0.0917
3m 8s (- 6m 30s) (65 32%) 0.0898
3m 22s (- 6m 15s) (70 35%) 0.0845
3m 36s (- 6m 1s) (75 37%) 0.0776
3m 51s (- 5m 47s) (80 40%) 0.0758
4m 5s (- 5m 32s) (85 42%) 0.0726
4m 20s (- 5m 18s) (90 45%) 0.0721
4m 35s (- 5m 4s) (95 47%) 0.0719
4m 50s (- 4m 50s) (100 50%) 0.0674
5m 5s (- 4m 36s) (105 52%) 0.0692
5m 20s (- 4m 22s) (110 55%) 0.0669
5m 35s (- 4m 7s) (115 57%) 0.0646
5m 50s (- 3m 53s) (120 60%) 0.0622
6m 4s (- 3m 38s) (125 62%) 0.0602
6m 19s (- 3m 24s) (130 65%) 0.061

In [20]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> c est un rouspeteur
= he s a grouch
< he s a grouch <EOS>

> nous sommes satisfaites
= we re contented
< we re almost back <EOS>

> vous etes branche
= you re fashionable
< you re fashionable <EOS>

> nous sommes armes
= we re armed
< we re armed <EOS>

> vous etes tres craintive
= you re very timid
< you re very timid <EOS>

> ce sont mes amis
= they are my friends
< they are my friends <EOS>

> elles sont jetables
= they re disposable
< they re disposable <EOS>

> tu es lunatique
= you re moody
< you are hilarious <EOS>

> vous etes fort contrarie
= you re very upset
< you re very upset <EOS>

> il est incroyablement idiot
= he s incredibly stupid
< he s incredibly stupid <EOS>



#### Discussion 

In this experiment, Luong Multiplicative Attention was implemented for Neural Machine Translation using an Encoder-Decoder model. The attention mechanism helped the decoder focus on relevant parts of the input sequence while generating translations. The model was trained on the English-French dataset using teacher forcing for improved learning.


#### Conclusion


The Luong Multiplicative Attention model was successfully implemented and trained for machine translation. It efficiently captures contextual information and improves translation performance compared to basic Seq2Seq models without attention.
